# LangGraph, smallest possible version

Goal: one `TypedDict` state, two nodes that each return a partial state update, one edge between them. Compile it, look at it, run it. That's the whole vocabulary LangGraph adds on top of "a function that takes state and returns state" — everything later (conditional edges, cycles, the repair loop) is this same shape, just wired differently.

In [1]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class GreetState(TypedDict):
    name: str
    greeting: str
    shout: str

A node is just a function: `state in -> partial state out`. It does **not** return the whole state — whatever keys it returns get merged into the shared state. That's why every node stays independently callable and testable: it's a plain function, no framework magic to invoke it by hand.

In [2]:
def make_greeting(state: GreetState) -> dict:
    return {"greeting": f"Hello, {state['name']}!"}


def shout_it(state: GreetState) -> dict:
    return {"shout": state["greeting"].upper()}


# proof these are plain functions - no graph needed to call them
shout_it({"name": "Mehul", "greeting": "jfdjs", "shout": ""})

{'shout': 'JFDJS'}

Now wire them into a graph: two nodes, one edge, an explicit entry (`START`) and exit (`END`).

In [3]:
builder = StateGraph(GreetState)
builder.add_node("make_greeting", make_greeting)
builder.add_node("shout_it", shout_it)

builder.add_edge(START, "make_greeting")
builder.add_edge("make_greeting", "shout_it")
builder.add_edge("shout_it", END)

graph = builder.compile()

Print the graph **before** running anything — the whole point of LangGraph over a `while` loop is that the graph is data you can inspect, not just behavior you have to trace by reading code.

In [7]:
print(graph.get_graph().draw_ascii())

  +-----------+    
  | __start__ |    
  +-----------+    
        *          
        *          
        *          
+---------------+  
| make_greeting |  
+---------------+  
        *          
        *          
        *          
  +----------+     
  | shout_it |     
  +----------+     
        *          
        *          
        *          
   +---------+     
   | __end__ |     
   +---------+     


In [8]:
result = graph.invoke({"name": "Mehul", "greeting": "", "shout": ""})
result

{'name': 'Mehul', 'greeting': 'Hello, Mehul!', 'shout': 'HELLO, MEHUL!'}

In [9]:
assert result["greeting"] == "Hello, Mehul!"
assert result["shout"] == "HELLO, MEHUL!"
print("ok")

ok
